# Notebook 4 — Stage 2: Transform Data (GPT Format → Qwen Format)
**MAI 656 — Natural Language Processing | Canadian University Dubai | Spring 2026**

This notebook converts the GPT-labeled JSONL file into the **Qwen2.5-VL chat format** required for LoRA fine-tuning, then splits into train/eval sets.

> 💡 **No GPU required.** Run this on local machine or free Colab.

**Input:** `data/gpt_labels.jsonl` (from Notebook 3)  
**Output:**
- `data/train/train.jsonl` — 80% of samples
- `data/eval/eval.jsonl` — 20% of samples

---

### Why Transform?

GPT receives images directly. Qwen2.5-VL expects a **multi-turn chat format** where the assistant's answer is part of the message list:

```json
{
  "messages": [
    {"role": "system",    "content": "You are an Arabic handwriting recognition system..."},
    {"role": "user",      "content": [{"type": "image_url", "image_url": {"url": "data:image/png;base64,..."}}]},
    {"role": "assistant", "content": "{\"transcription\": \"...\", \"confidence\": \"high\"}"}
  ]
}
```

## 1. Install Dependencies

In [3]:
%pip install Pillow 

Note: you may need to restart the kernel to use updated packages.


## 2. Set Project Root

**Colab:** mounts Google Drive and reads from `MyDrive/nlp_project`.  
**Local:** update `LOCAL_PROJECT_ROOT` below to point to your project folder.

In [4]:
import os
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/nlp_project')
else:
    # ── Set this to your local project directory ──────────────────────────
    PROJECT_ROOT = Path(r'C:\Users\mitah\github_projects\nlp_project')
    # ──────────────────────────────────────────────────────────────────────

assert PROJECT_ROOT.exists(), (
    f'Project root not found: {PROJECT_ROOT}\n'
    'Update LOCAL_PROJECT_ROOT above to match your system.'
)

print(f'{"Colab" if IN_COLAB else "Local"} | Project root: {PROJECT_ROOT}')

Local | Project root: C:\Users\mitah\github_projects\nlp_project


## 3. Configure Paths & Constants

> ⚠️ `MIN_PIXELS`, `MAX_PIXELS`, and `DPI` **must match** the values used during training and inference.

In [11]:
from pathlib import Path

# Paths
INPUT_FILE = Path(PROJECT_ROOT) / 'sample_training_data/gpt_output/raw_png.jsonl'
TRAIN_FILE = Path(PROJECT_ROOT) / 'sample_training_data/train/train.jsonl'
EVAL_FILE  = Path(PROJECT_ROOT) / 'sample_training_data/eval/eval.jsonl'

# Image pixel bounds — MUST match training & inference!
MIN_PIXELS = 4   * 28 * 28   # 3,136
MAX_PIXELS = 128 * 28 * 28   # 100,352
DPI = 100

# Train/eval split ratio
TRAIN_RATIO = 0.8
RANDOM_SEED = 42

# Create output dirs
TRAIN_FILE.parent.mkdir(parents=True, exist_ok=True)
EVAL_FILE.parent.mkdir(parents=True, exist_ok=True)

print(f'Input:  {INPUT_FILE}')
print(f'Train:  {TRAIN_FILE}')
print(f'Eval:   {EVAL_FILE}')

Input:  C:\Users\mitah\github_projects\nlp_project\sample_training_data\gpt_output\raw_png.jsonl
Train:  C:\Users\mitah\github_projects\nlp_project\sample_training_data\train\train.jsonl
Eval:   C:\Users\mitah\github_projects\nlp_project\sample_training_data\eval\eval.jsonl


## 4. Define Transformation Functions

In [7]:
import json
import base64
from PIL import Image
import io

def simplify_gpt_output(raw_response: str) -> str:
    """Strip GPT's output to essential fields only."""
    try:
        data = json.loads(raw_response)
        simplified = {
            'transcription': data.get('transcription', ''),
            'confidence': data.get('confidence', 'unknown')
        }
        return json.dumps(simplified, ensure_ascii=False)
    except json.JSONDecodeError:
        return None


def image_to_base64(image_path: str) -> str:
    """Convert an image to a base64 PNG string, respecting pixel bounds."""
    img = Image.open(image_path)
    w, h = img.size
    total_pixels = w * h

    if total_pixels > MAX_PIXELS:
        scale = (MAX_PIXELS / total_pixels) ** 0.5
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    elif total_pixels < MIN_PIXELS:
        scale = (MIN_PIXELS / total_pixels) ** 0.5
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)

    buffer = io.BytesIO()
    img.save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode('utf-8')


def transform_sample(sample: dict) -> dict:
    """Convert a GPT-labeled sample to Qwen training format."""
    image_path = sample['image_path']
    gpt_response = sample['gpt_response']

    simplified = simplify_gpt_output(gpt_response)
    if simplified is None:
        return None

    try:
        img_b64 = image_to_base64(image_path)
    except Exception as e:
        print(f'  Image load error ({image_path}): {e}')
        return None

    return {
        'messages': [
            {'role': 'system',
             'content': 'You are an Arabic handwriting recognition system. '
                        'Read the handwritten Arabic text in this image and '
                        'return the transcription as JSON.'},
            {'role': 'user', 'content': [
                {'type': 'image_url',
                 'image_url': {'url': f'data:image/png;base64,{img_b64}'}}
            ]},
            {'role': 'assistant', 'content': simplified}
        ]
    }

print('Transform functions defined.')

Transform functions defined.


## 5. Run the Transformation

In [9]:
# Load all GPT-labeled samples
raw_samples = []
with open(INPUT_FILE, encoding='utf-8') as f:
    for line in f:
        raw_samples.append(json.loads(line))

print(f'Loaded {len(raw_samples)} raw samples')

# Transform
samples = []
skipped = 0
for i, raw in enumerate(raw_samples):
    result = transform_sample(raw)
    if result:
        samples.append(result)
    else:
        skipped += 1

print(f'Successfully transformed: {len(samples)}')
print(f'Skipped (bad JSON or missing image): {skipped}')

Loaded 100 raw samples
Successfully transformed: 100
Skipped (bad JSON or missing image): 0


## 6. Split into Train / Eval and Save

In [12]:
import random

random.seed(RANDOM_SEED)
random.shuffle(samples)

split = int(len(samples) * TRAIN_RATIO)
train_samples = samples[:split]
eval_samples  = samples[split:]

for filepath, data in [(TRAIN_FILE, train_samples), (EVAL_FILE, eval_samples)]:
    with open(filepath, 'w', encoding='utf-8') as f:
        for s in data:
            f.write(json.dumps(s, ensure_ascii=False) + '\n')

print(f'Train samples: {len(train_samples)} → {TRAIN_FILE}')
print(f'Eval samples:  {len(eval_samples)}  → {EVAL_FILE}')

Train samples: 80 → C:\Users\mitah\github_projects\nlp_project\sample_training_data\train\train.jsonl
Eval samples:  20  → C:\Users\mitah\github_projects\nlp_project\sample_training_data\eval\eval.jsonl


## 7. Verify the Output

In [13]:
# Inspect one training sample (without printing the huge base64 image)
with open(TRAIN_FILE) as f:
    sample = json.loads(f.readline())

print('=== Sample message structure ===')
for msg in sample['messages']:
    role = msg['role']
    if isinstance(msg['content'], list):
        content_preview = f'[image_url — base64 len={len(msg["content"][0]["image_url"]["url"])}]'
    else:
        content_preview = msg['content'][:120]
    print(f'  [{role}] {content_preview}')

=== Sample message structure ===
  [system] You are an Arabic handwriting recognition system. Read the handwritten Arabic text in this image and return the transcri
  [user] [image_url — base64 len=5410]
  [assistant] {"transcription": "Ø§Ù„Ù…Ø¹Ø±Ø¶ØŒ Ø£Ùˆ ØªÙ‚ÙˆÙ… Ø¨Ù‡ØŒ", "confidence": "medium"}


## Next Step

Train/eval JSONL files are ready. Continue to:

**Notebook 5 → `05_finetuning.ipynb`** — LoRA fine-tuning on the GPU instance (g5.xlarge required)